# baseline v3 (로컬 GPU 버전)

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

원래 Colab(T4 GPU)용으로 개발된 코드를 **로컬 Windows PC의 NVIDIA GPU**에서 실행되도록 수정했습니다.

#### Colab 버전과 달라진 점
| 항목 | Colab 원본 | 로컬 버전 |
|---|---|---|
| PyTorch | CUDA 12.1 빌드 (`cu121`) | CUDA 12.8 빌드 (`cu128`) — RTX 50 시리즈(Blackwell, sm_120)는 cu128 이상이 필요 |
| 패키지 설치 | `!pip` | `%pip` (현재 커널의 가상환경에 설치) |
| 데이터 위치 | 구글 드라이브 마운트 | 로컬 폴더 경로 (`DATA_PATH`) |
| 정밀도 | fp16 양자화 연산 + bf16 autocast 혼용 | GPU가 지원하면 bf16으로 통일, GradScaler는 fp16일 때만 사용 |
| 모델 배치 | `device_map="auto"` + `.to(device)` | `device_map={"": 0}` (GPU 0번에 전부 올림, CPU 오프로딩 방지) |
| 저장 경로 | `/content/...` | `./outputs/...` |

#### 실행 전 준비
- VS Code 우측 상단 커널 선택에서 이 프로젝트의 `.venv` (Python 3.13)를 선택하세요.
- NVIDIA 드라이버가 설치되어 있어야 합니다. (터미널에서 `nvidia-smi`로 확인)

# 환경 준비

로컬 가상환경(`.venv`)에 필요한 라이브러리를 설치합니다. **처음 한 번만** 실행하면 됩니다.

- 아래 셀 실행 (PyTorch 다운로드 약 3GB)
- 실행 완료 후 VS Code 노트북 상단의 **Restart(커널 다시 시작)** 클릭
- `transformers`는 GitHub 최신 소스에서 설치하므로 `git`이 필요합니다.

In [29]:
# RTX 50 시리즈(Blackwell)는 CUDA 12.8 이상으로 빌드된 PyTorch가 필요합니다.
%pip install -q --index-url https://download.pytorch.org/whl/cu128 torch torchvision
%pip install -q git+https://github.com/huggingface/transformers accelerate
# bitsandbytes는 0.46 이상부터 Windows + CUDA 12.8 휠을 제공합니다.
# ipywidgets는 노트북에서 tqdm 진행바를 표시하는 데 필요합니다.
%pip install -q "peft>=0.13.2" "bitsandbytes>=0.46.1" datasets pillow pandas ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [30]:
# GPU 환경 점검
import torch

print("Torch version :", torch.__version__)
print("CUDA (torch)  :", torch.version.cuda)
print("cuDNN version :", torch.backends.cudnn.version())
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU           :", props.name)
    print("VRAM          :", f"{props.total_memory / 1024**3:.1f} GB")
    print("Compute cap.  :", f"sm_{major}{minor}")
    print("bf16 지원     :", torch.cuda.is_bf16_supported())

    # 설치된 PyTorch가 이 GPU 아키텍처용 커널을 포함하는지 확인
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("\n[경고] 설치된 PyTorch가 이 GPU를 지원하지 않습니다. 위 설치 셀의 cu128 빌드를 확인하세요.")

    # 간단한 연산 테스트
    x = torch.randn(1024, 1024, device="cuda")
    print("GPU 연산 테스트:", "OK" if torch.isfinite(x @ x).all().item() else "FAIL")
else:
    print("\n[경고] GPU를 찾지 못했습니다. 드라이버(nvidia-smi)와 PyTorch CUDA 빌드를 확인하세요.")

Torch version : 2.11.0+cu128
CUDA (torch)  : 12.8
cuDNN version : 91900
CUDA 사용 가능: True
GPU           : NVIDIA GeForce RTX 5060 Ti
VRAM          : 15.9 GB
Compute cap.  : sm_120
bf16 지원     : True
GPU 연산 테스트: OK


# 데이터 준비

개발에 필요한 데이터를 로컬 폴더에 준비합니다. 기본 경로는 노트북과 같은 폴더 아래의 `data` 폴더입니다.

```
mosquito_AI/
├─ (260908)_baseline_local.ipynb
└─ data/
   ├─ train.csv   (id, path, question, a, b, c, d, answer)
   ├─ dev.csv
   ├─ test.csv    (id, path, question, a, b, c, d)
   └─ (이미지 파일들 — csv의 path 열은 data 폴더 기준 상대경로)
```

다른 위치에 데이터가 있다면 아래 셀의 `DATA_PATH`만 바꾸면 됩니다.
Colab의 구글 드라이브 마운트 셀은 로컬에서는 필요 없으므로 삭제했습니다.

# 라이브러리, 데이터, 설정

In [31]:
import os, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda:0")
SEED = 42
random.seed(SEED); np.random.seed(SEED); 
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED); 
torch.backends.cudnn.deterministic = True; 
torch.backends.cudnn.benchmark = False

DATA_PATH  = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

labels = pd.read_csv(DATA_PATH / "train_labels.csv")      # id, x, y, z (+80ms 정답)
sub    = pd.read_csv(DATA_PATH / "sample_submission.csv")  # id, x, y, z

def load_tracks(folder, ids):
    # 파일 하나 = 11개 시점 (timestep_ms -400 ~ 0)
    tracks = []
    for i in tqdm(ids, desc=folder.name):
        df = pd.read_csv(folder / f"{i}.csv").sort_values("timestep_ms")
        tracks.append(df[["x", "y", "z"]].to_numpy())
    return np.stack(tracks).astype(np.float32)

X_train = load_tracks(DATA_PATH / "train", labels["id"])   # (10000, 11, 3)
Y_train = labels[["x", "y", "z"]].to_numpy(np.float32)     # (10000, 3)
X_test  = load_tracks(DATA_PATH / "test", sub["id"])       # (10000, 11, 3)


train:   0%|          | 0/10000 [00:00<?, ?it/s]

test:   0%|          | 0/10000 [00:00<?, ?it/s]

## 입력 특징과 정답 만들기
학습 데이터와 평가 데이터는 장소가 다릅니다. 그래서 절대 좌표 대신 이동량을 입력으로 씁니다. 미터 단위는 값이 너무 작아서 100을 곱해 센티미터 단위로 학습합니다.

In [32]:
SCALE = 100.0  # m -> cm

def make_features(X):
    vel = np.diff(X, axis=1)        # (N, 10, 3): 40ms마다의 이동량
    # vel = (X - X[:, -1:])             # 마지막 점 기준 상대 좌표 / 이동량이 더 값이 좋게 나옴
    return vel * SCALE

def baseline(X):
    # 등속 외삽: 80ms 뒤 = 마지막 위치 + 마지막 40ms 이동량 x 2
    return X[:, -1] + 2 * (X[:, -1] - X[:, -2])

feats_train = make_features(X_train)
resid_train = (Y_train - baseline(X_train)) * SCALE   # 모델이 맞힐 대상 = 기준선의 오차
feats_test  = make_features(X_test)

## Dataset
z축을 기준으로 무작위 회전시키는 증강이 일반화에 가장 효과가 컸습니다. 입력과 정답을 반드시 같은 각도로 함께 돌려야 합니다. z축은 위쪽 방향이라 회전하지 않습니다.

In [33]:
class TrajDataset(Dataset):
    def __init__(self, feats, targets=None, augment=False):
        self.feats, self.targets, self.augment = feats, targets, augment

    def __len__(self): return len(self.feats)

    def __getitem__(self, i):
        v = torch.from_numpy(self.feats[i])                       # (10, 3)
        t = torch.from_numpy(self.targets[i]) if self.targets is not None else torch.zeros(3)
        if self.augment:
            th = random.uniform(0, 2 * math.pi)
            c, s = math.cos(th), math.sin(th)
            R = torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32)
            v, t = v @ R.T, t @ R.T
        return v.reshape(-1), t                                   # (30,), (3,)

idx = np.random.permutation(len(X_train))
tr_idx, va_idx = idx[:9000], idx[9000:]

train_loader = DataLoader(TrajDataset(feats_train[tr_idx], resid_train[tr_idx], augment=True),
                          batch_size=256, shuffle=True, num_workers=0)
valid_loader = DataLoader(TrajDataset(feats_train[va_idx], resid_train[va_idx]),
                          batch_size=1024, shuffle=False, num_workers=0)

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 처음 한 번만 받고, 이후에는 로컬 캐시(`%USERPROFILE%\.cache\huggingface`)에서 불러옵니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [34]:
class TrajMLP(nn.Module):
    
    def __init__(self, in_dim=30, hidden=128):
        
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 3),
        )
    def forward(self, x): return self.net(x)

in_dim = feats_train.shape[1] * feats_train.shape[2]   # 상대 좌표면 11×3 = 33
model = TrajMLP(in_dim=in_dim).to(device)
print("입력 크기:", in_dim)


입력 크기: 30


# fine-tuning

#### 참고

    손실은 평가 기준과 같은 3D 유클리드 거리로 둡니다. 모델이 작아서 bf16, GradScaler, gradient 누적은 필요 없습니다.

In [35]:
N_FOLDS    = 5
EPOCHS     = 200
FT_EPOCHS  = 50
USE_FT     = True
HIT_RADIUS = 0.01   # m

# 샘플마다 0~4 중 하나의 fold 번호를 붙임 (fold마다 2000개)
perm  = np.random.RandomState(SEED).permutation(len(X_train))
folds = np.empty(len(X_train), dtype=int)
folds[perm] = np.arange(len(X_train)) % N_FOLDS
print("fold별 샘플 수:", np.bincount(folds))


fold별 샘플 수: [2000 2000 2000 2000 2000]


In [36]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def dist3d(pred, target):
    return torch.linalg.norm(pred - target, dim=1)

def hit_loss(pred, target, radius=1.0, tau=0.15, dist_weight=0.1):
    d = dist3d(pred, target)
    return (1 - torch.sigmoid((radius - d) / tau)).mean() + dist_weight * d.mean()

def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        d = torch.cat([dist3d(model(v.to(device)), t.to(device)) for v, t in loader]) / SCALE  # m
    return d.mean().item(), (d < HIT_RADIUS).float().mean().item()

def train_stage(model, train_loader, valid_loader, loss_fn, epochs, lr, save_path, keep_best, desc):
    """keep_best=True: 검증 평균 오차가 가장 낮은 epoch 저장 / False: 마지막 epoch 저장"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    best = float("inf")
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for v, t in train_loader:
            v, t = v.to(device), t.to(device)
            loss = loss_fn(model(v), t)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if keep_best:
            val_dist, _ = evaluate(model, valid_loader)
            if val_dist < best:
                best = val_dist
                torch.save(model.state_dict(), save_path)
    if not keep_best:
        torch.save(model.state_dict(), save_path)
    model.load_state_dict(torch.load(save_path))   # 저장한 시점의 모델로 되돌림
    return model

def predict(model, feats):
    """보정값 예측, cm 단위"""
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(feats.reshape(len(feats), -1)).to(device)
        return model(x).cpu().numpy()

def report(name, pred_cm, idx=None):
    idx = np.arange(len(resid_train)) if idx is None else idx
    d = np.linalg.norm(pred_cm - resid_train[idx], axis=1) / SCALE
    print(f"{name:12s} 평균 오차 {d.mean()*100:.3f}cm | 명중률 {(d < HIT_RADIUS).mean()*100:.2f}%")


In [37]:
in_dim = feats_train.shape[1] * feats_train.shape[2]

oof_base  = np.zeros_like(resid_train)                   # 각 샘플이 검증 fold일 때의 예측 (cm)
oof_ft    = np.zeros_like(resid_train)
test_base = np.zeros((len(X_test), 3), dtype=np.float32)  # fold 모델 5개의 테스트 예측 평균 (cm)
test_ft   = np.zeros((len(X_test), 3), dtype=np.float32)

for k in range(N_FOLDS):
    set_seed(SEED + k)
    tr_idx, va_idx = np.where(folds != k)[0], np.where(folds == k)[0]

    train_loader = DataLoader(TrajDataset(feats_train[tr_idx], resid_train[tr_idx], augment=True),
                              batch_size=256, shuffle=True, num_workers=0)
    valid_loader = DataLoader(TrajDataset(feats_train[va_idx], resid_train[va_idx]),
                              batch_size=1024, shuffle=False, num_workers=0)

    # 1단계: 거리 손실
    model = TrajMLP(in_dim=in_dim).to(device)
    model = train_stage(model, train_loader, valid_loader, lambda p, t: dist3d(p, t).mean(),
                        EPOCHS, 1e-3, OUTPUT_DIR / f"fold{k}_base.pt", keep_best=True, desc=f"fold{k} base")
    oof_base[va_idx] = predict(model, feats_train[va_idx])
    test_base += predict(model, feats_test) / N_FOLDS

    # 2단계: 명중률 손실로 추가 학습
    if USE_FT:
        model = train_stage(model, train_loader, valid_loader, hit_loss,
                            FT_EPOCHS, 1e-4, OUTPUT_DIR / f"fold{k}_ft.pt", keep_best=False, desc=f"fold{k} ft")
        oof_ft[va_idx] = predict(model, feats_train[va_idx])
        test_ft += predict(model, feats_test) / N_FOLDS

    print(f"--- fold {k} ---")
    report("base", oof_base[va_idx], va_idx)
    if USE_FT:
        report("ft",  oof_ft[va_idx], va_idx)
        report("avg", (oof_base[va_idx] + oof_ft[va_idx]) / 2, va_idx)


fold0 base:   0%|          | 0/200 [00:00<?, ?it/s]

fold0 ft:   0%|          | 0/50 [00:00<?, ?it/s]

--- fold 0 ---
base         평균 오차 1.178cm | 명중률 64.85%
ft           평균 오차 1.179cm | 명중률 65.10%
avg          평균 오차 1.178cm | 명중률 65.10%


fold1 base:   0%|          | 0/200 [00:00<?, ?it/s]

fold1 ft:   0%|          | 0/50 [00:00<?, ?it/s]

...

fold3 ft:   0%|          | 0/50 [00:00<?, ?it/s]

--- fold 3 ---
base         평균 오차 1.233cm | 명중률 63.70%
ft           평균 오차 1.233cm | 명중률 64.00%
avg          평균 오차 1.232cm | 명중률 63.90%


fold4 base:   0%|          | 0/200 [00:00<?, ?it/s]

fold4 ft:   0%|          | 0/50 [00:00<?, ?it/s]

--- fold 4 ---
base         평균 오차 1.242cm | 명중률 63.50%
ft           평균 오차 1.247cm | 명중률 63.60%
avg          평균 오차 1.243cm | 명중률 63.70%


# inference

In [38]:
print("=== 전체 OOF (학습 데이터 10000개) ===")
report("등속 외삽", np.zeros_like(resid_train))
report("base", oof_base)
if USE_FT:
    report("ft",  oof_ft)
    report("avg", (oof_base + oof_ft) / 2)


=== 전체 OOF (학습 데이터 10000개) ===
등속 외삽        평균 오차 1.294cm | 명중률 57.87%
base         평균 오차 1.192cm | 명중률 64.15%
ft           평균 오차 1.194cm | 명중률 64.39%
avg          평균 오차 1.192cm | 명중률 64.35%


In [39]:
resid_test = (test_base + test_ft) / 2 if USE_FT else test_base
sub[["x", "y", "z"]] = baseline(X_test) + resid_test / SCALE
sub.to_csv(OUTPUT_DIR / "submission_kfold.csv", index=False)
print(sub.head())


           id         x         y         z
0  TEST_00001  3.988665 -1.052376  0.045575
1  TEST_00002  1.927984  0.618386  0.798082
2  TEST_00003  0.805559 -0.190384  0.148137
3  TEST_00004  3.525990 -1.646430 -0.941414
4  TEST_00005  2.304393  0.017422 -0.039712


# (선택) GPU 메모리 정리

다른 실험을 이어서 하기 전에 VRAM을 비우고 싶을 때 실행합니다. 커널 재시작이 가장 확실한 방법입니다.

In [40]:
import gc
del model, optimizer
gc.collect()
torch.cuda.empty_cache()
print(f"현재 VRAM 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

NameError: name 'optimizer' is not defined